# Home Credit — Evidence Parameter Sensitivity Audit

**目的：** 评估类别压缩及 IV 分组参数对特征排序的影响，避免后续特征选择依赖未经检验的人工阈值。

本补充实验依赖 Notebook 7 的预处理规则和 Notebook 8 的特征证据。代码只使用 Train（Week 0–63），不读取 Validation（Week 64–77）或 OOT（Week 78–91）。

| 审查对象 | 基准设置 | 对照设置或指标 |
|---|---|---|
| 类别压缩（NB7） | 最小类别频数 100、最多约 100 类 | `rare_by_frequency_share`、`cap_collapsed_share` |
| Categorical IV（NB8） | 最多 20 个类别 | 40 / 100 类、IV 排名与 Relevance 排名 |
| 排名稳健性 | 基准 Relevance | Spearman、Top-300 / Top-500 Jaccard、排名变动 |

本 Notebook 属于上游证据的参数敏感性诊断，不直接重训最终预测模型，也不将任意 Jaccard 数值当成统一的通过阈值。


## 1. 审查范围与计算策略

先检查少量高影响参数是否改变特征排序，而非重新计算完整的模型证据或穷举全部阈值。输出分别展示类别合并的样本占比，以及特征 Top-k 成员对分组分辨率的敏感性。


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42

WEEK_COL = "WEEK_NUM"
TARGET_COL = "target"

MISSING_TOKEN = "__HC_MISSING__"
RARE_TOKEN = "__HC_RARE__"
OTHER_TOKEN = "__OTHER__"

# ------------------------------------------------------------
# Notebook 7 sensitivity settings
# ------------------------------------------------------------
NB7_SETTINGS = [
    ("min50_cap100", 50, 100),
    ("current_100_100", 100, 100),
    ("min200_cap100", 200, 100),
    ("min100_cap50", 100, 50),
    ("min100_cap200", 100, 200),
]

# ------------------------------------------------------------
# Notebook 8 categorical IV sensitivity
# ------------------------------------------------------------
IV_CATEGORY_SETTINGS = [20, 40, 100]
IV_RULE_SAMPLE_SIZE = 200_000
IV_SMOOTHING = 0.5

TOP_K_CHECKS = [300, 500]


def find_one(filename):
    roots = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
        Path("/mnt/data"),
        Path("."),
    ]

    matches = []

    for root in roots:
        if not root.exists():
            continue

        try:
            matches.extend(root.rglob(filename))
        except Exception:
            pass

    if not matches:
        raise FileNotFoundError(
            f"找不到 {filename}。请把对应 Notebook 输出加入当前环境。"
        )

    # 优先 working，其次较短路径
    matches = sorted(
        set(matches),
        key=lambda p: (
            0 if "/kaggle/working/" in str(p) else 1,
            len(str(p)),
        ),
    )

    print(filename, "->", matches[0])
    return matches[0]


TRAIN_PATH = find_one("train_clean.parquet")
NB7_CATALOG_PATH = find_one("nb7_candidate_features.csv")
NB8_EVIDENCE_PATH = find_one("feature_evidence_report.csv")

catalog = pd.read_csv(NB7_CATALOG_PATH)
evidence = pd.read_csv(NB8_EVIDENCE_PATH)

required_catalog = {"feature", "dtype_family"}
required_evidence = {
    "feature",
    "iv",
    "mi_percentile",
    "relevance_score",
}

assert required_catalog.issubset(catalog.columns)
assert required_evidence.issubset(evidence.columns)

categorical_features = (
    catalog.loc[
        catalog["dtype_family"].isin(["categorical", "boolean"]),
        "feature",
    ]
    .drop_duplicates()
    .tolist()
)

print("Categorical / Boolean features:", len(categorical_features))
print("Evidence features:", evidence["feature"].nunique())

train_clean.parquet -> /kaggle/input/notebooks/jing78y38/5-pandas-zjing/clean_split/train_clean.parquet
nb7_candidate_features.csv -> /kaggle/input/notebooks/jing78y38/7-zjing/reports/reference_model/nb7_candidate_features.csv
feature_evidence_report.csv -> /kaggle/input/notebooks/jing78y38/8-zjing/reports/feature_evidence/feature_evidence_report.csv
Categorical / Boolean features: 59
Evidence features: 1209


## 2. 类别压缩占比审查

Notebook 7 的预处理同时包含低频合并（`count < 100`）和类别数量上限（约 100 个最终状态，包括 Missing、Rare）。

本节分开统计两类影响：一是低频规则合并的样本比例，二是类别上限额外合并的样本比例。仅凭原始类别数量无法判断实际的信息损失；应结合样本覆盖情况进行解释。


In [2]:
def category_compression_summary(series, min_frequency, max_categories):
    values = (
        series.astype("string")
        .fillna(MISSING_TOKEN)
    )

    counts = values.value_counts(dropna=False)

    missing_count = int(counts.get(MISSING_TOKEN, 0))

    normal_counts = (
        counts.drop(
            labels=[MISSING_TOKEN],
            errors="ignore",
        )
        .sort_values(ascending=False)
    )

    frequent_counts = normal_counts[
        normal_counts >= min_frequency
    ]

    normal_limit = max(1, max_categories - 2)

    kept_counts = frequent_counts.head(normal_limit)

    n = len(values)
    non_missing_rows = int(normal_counts.sum())
    frequent_rows = int(frequent_counts.sum())
    kept_rows = int(kept_counts.sum())

    return {
        "raw_unique_count": int(values.nunique(dropna=False)),
        "missing_share": missing_count / n,
        "rare_by_frequency_share": (
            (non_missing_rows - frequent_rows) / n
        ),
        "cap_collapsed_share": (
            (frequent_rows - kept_rows) / n
        ),
        "kept_normal_share": kept_rows / n,
        "frequent_normal_count": int(len(frequent_counts)),
        "kept_normal_count": int(len(kept_counts)),
    }


nb7_rows = []

BATCH_SIZE = 8

for start in range(
    0,
    len(categorical_features),
    BATCH_SIZE,
):
    batch = categorical_features[
        start:start + BATCH_SIZE
    ]

    frame = pd.read_parquet(
        TRAIN_PATH,
        columns=batch,
        engine="pyarrow",
    )

    for feature in batch:
        for setting_name, min_frequency, max_categories in NB7_SETTINGS:
            row = category_compression_summary(
                frame[feature],
                min_frequency=min_frequency,
                max_categories=max_categories,
            )

            row.update({
                "feature": feature,
                "setting": setting_name,
                "min_frequency": min_frequency,
                "max_categories": max_categories,
            })

            nb7_rows.append(row)

    del frame

nb7_sensitivity = pd.DataFrame(nb7_rows)

current_nb7 = (
    nb7_sensitivity
    .loc[
        nb7_sensitivity["setting"].eq("current_100_100")
    ]
    .copy()
)

display(
    current_nb7
    .sort_values(
        [
            "cap_collapsed_share",
            "rare_by_frequency_share",
        ],
        ascending=False,
    )
    .head(30)
)

,raw_unique_count,missing_share,rare_by_frequency_share,cap_collapsed_share,kept_normal_share,frequent_normal_count,kept_normal_count,feature,setting,min_frequency,max_categories
191,3510,0.000000,6.644661e-02,0.278012,0.655542,1202,98,person_main_registaddr_zipcode_184M,current_100_100,100,100
136,3504,0.000000,6.608548e-02,0.273375,0.660540,1187,98,person_main_contaddr_zipcode_807M,current_100_100,100,100
186,689,0.000000,2.908550e-03,0.147487,0.849605,223,98,person_main_registaddr_district_1083M,current_100_100,100,100
126,671,0.000000,2.680016e-03,0.147480,0.849840,224,98,person_main_contaddr_district_15M,current_100_100,100,100
76,223,0.000000,2.254146e-04,0.094987,0.904788,200,98,previouscontdistrict_112M,current_100_100,100,100
161,3324,0.000000,4.650015e-02,0.047304,0.906196,354,98,person_main_empladdr_zipcode_114M,current_100_100,100,100
156,223,0.000000,2.246346e-04,0.047106,0.952669,217,98,person_main_empladdr_district_926M,current_100_100,100,100
291,105476,0.653826,2.479545e-01,0.039604,0.058616,428,98,taxa_top_employer_code,current_100_100,100,100
46,68,0.000000,9.156981e-04,0.000000,0.999084,30,30,lastcancelreason_561M,current_100_100,100,100
41,45,0.000000,5.015280e-04,0.000000,0.999498,25,25,lastapprcommoditycat_1041M,current_100_100,100,100


In [3]:
# 把几组设置横向放在一起。
# 这里不自动给“通过 / 不通过”，先看真实变化。

nb7_compare = (
    nb7_sensitivity
    .pivot_table(
        index="feature",
        columns="setting",
        values=[
            "rare_by_frequency_share",
            "cap_collapsed_share",
        ],
        aggfunc="first",
    )
)

display(nb7_compare.head(20))

print("\nCurrent 100/100 summary")
display(
    current_nb7[
        [
            "raw_unique_count",
            "rare_by_frequency_share",
            "cap_collapsed_share",
            "kept_normal_share",
        ]
    ]
    .describe()
    .T
)

nb7_sensitivity.to_csv(
    "nb7_category_threshold_sensitivity.csv",
    index=False,
)

cap_collapsed_share                \
setting                             current_100_100 min100_cap200   
feature                                                             
cardtype_51L                                    0.0           0.0   
credtype_322L                                   0.0           0.0   
disbursementtype_67L                            0.0           0.0   
education_1103M                                 0.0           0.0   
education_88M                                   0.0           0.0   
equalitydataagreement_891L                      0.0           0.0   
equalityempfrom_62L                             0.0           0.0   
inittransactioncode_186L                        0.0           0.0   
isbidproduct_1095L                              0.0           0.0   
isbidproductrequest_292L                        0.0           0.0   
last_credtype_587L                              0.0           0.0   
last_status_219L                                0.0           0.0   
lastapprcommoditycat_1041M                      0.0           0.0   
lastcancelreason_561M                           0.0           0.0   
lastrejectcommoditycat_161M                     0.0           0.0   
lastrejectreason_759M                           0.0           0.0   
lastrejectreasonclient_4145040M                 0.0           0.0   
lastst_736L                                     0.0           0.0   
maritalst_385M                                  0.0           0.0   
maritalst_893M                                  0.0           0.0   

                                                                         \
setting                         min100_cap50 min200_cap100 min50_cap100   
feature                                                                   
cardtype_51L                             0.0           0.0          0.0   
credtype_322L                            0.0           0.0          0.0   
disbursementtype_67L                     0.0           0.0          0.0   
education_1103M                          0.0           0.0          0.0   
education_88M                            0.0           0.0          0.0   
equalitydataagreement_891L               0.0           0.0          0.0   
equalityempfrom_62L                      0.0           0.0          0.0   
inittransactioncode_186L                 0.0           0.0          0.0   
isbidproduct_1095L                       0.0           0.0          0.0   
isbidproductrequest_292L                 0.0           0.0          0.0   
last_credtype_587L                       0.0           0.0          0.0   
last_status_219L                         0.0           0.0          0.0   
lastapprcommoditycat_1041M               0.0           0.0          0.0   
lastcancelreason_561M                    0.0           0.0          0.0   
lastrejectcommoditycat_161M              0.0           0.0          0.0   
lastrejectreason_759M                    0.0           0.0          0.0   
lastrejectreasonclient_4145040M          0.0           0.0          0.0   
lastst_736L                              0.0           0.0          0.0   
maritalst_385M                           0.0           0.0          0.0   
maritalst_893M                           0.0           0.0          0.0   

                                rare_by_frequency_share                \
setting                                 current_100_100 min100_cap200   
feature                                                                 
cardtype_51L                               0.000000e+00  0.000000e+00   
credtype_322L                              0.000000e+00  0.000000e+00   
disbursementtype_67L                       0.000000e+00  0.000000e+00   
education_1103M                            0.000000e+00  0.000000e+00   
education_88M                              6.395847e-05  6.395847e-05   
equalitydataagreement_891L                 0.000000e+00  0.000000e+00   
equalityempfrom_62L                        0.000000e+00  0.00


Current 100/100 summary


,count,mean,std,min,25%,50%,75%,max
raw_unique_count,59.0,2001.949153,13724.763648,2.0,4.000000,7.000000,12.000000,105476.000000
rare_by_frequency_share,59.0,0.007385,0.034548,0.0,0.000000,0.000000,0.000074,0.247954
cap_collapsed_share,59.0,0.018226,0.057382,0.0,0.000000,0.000000,0.000000,0.278012
kept_normal_share,59.0,0.594257,0.399674,0.0,0.136712,0.771217,0.999497,1.000000


## 3. Categorical IV 分组分辨率审查

比较 `IV_MAX_CATEGORIES = 20 / 40 / 100`。在同一份 Train 数据上使用固定抽样位置学习保留的类别集合，再对完整 Train 累计 Good / Bad 计数。

本节保持 MI 不变，只更新 IV、IV percentile、Relevance score 和相关排序，以隔离类别上限变化带来的影响；不将其解读为整个 Notebook 8 的完整敏感性复验。


In [4]:
meta = pd.read_parquet(
    TRAIN_PATH,
    columns=[WEEK_COL, TARGET_COL],
    engine="pyarrow",
).reset_index(drop=True)

assert meta[WEEK_COL].between(0, 63).all()

def stratified_positions(frame, sample_size, random_state):
    sample_size = min(sample_size, len(frame))

    rng = np.random.default_rng(random_state)

    groups = (
        frame
        .groupby([WEEK_COL, TARGET_COL], observed=True)
        .indices
    )

    pieces = []

    for key, positions in groups.items():
        positions = np.asarray(positions, dtype=np.int64)

        quota = max(
            1,
            int(round(
                len(positions) / len(frame) * sample_size
            )),
        )

        quota = min(quota, len(positions))

        pieces.append(
            rng.choice(
                positions,
                size=quota,
                replace=False,
            )
        )

    out = np.concatenate(pieces)

    if len(out) > sample_size:
        out = rng.choice(
            out,
            size=sample_size,
            replace=False,
        )

    return np.sort(out)


iv_rule_positions = stratified_positions(
    meta,
    IV_RULE_SAMPLE_SIZE,
    RANDOM_SEED + 10,
)

print("IV rule sample rows:", len(iv_rule_positions))

IV rule sample rows: 200000


In [5]:
def learn_kept_levels(series, positions, max_categories):
    sample_values = (
        series.iloc[positions]
        .astype("string")
        .fillna("__MISSING__")
    )

    return (
        sample_values
        .value_counts()
        .head(max_categories)
        .index
        .astype(str)
        .tolist()
    )


def calculate_full_iv_categorical(
    series,
    target,
    kept_levels,
    smoothing=0.5,
):
    values = (
        series.astype("string")
        .fillna("__MISSING__")
    )

    grouped = values.where(
        values.isin(kept_levels),
        OTHER_TOKEN,
    )

    table = pd.crosstab(
        grouped,
        target,
    )

    good = (
        table.get(
            0,
            pd.Series(0, index=table.index),
        ).astype(float)
        + smoothing
    )

    bad = (
        table.get(
            1,
            pd.Series(0, index=table.index),
        ).astype(float)
        + smoothing
    )

    good_dist = good / good.sum()
    bad_dist = bad / bad.sum()

    woe = np.log(good_dist / bad_dist)

    return float(
        ((good_dist - bad_dist) * woe).sum()
    )


# 只重算真正可能被 20-category cap 影响的 categorical features。
high_card_features = (
    current_nb7
    .loc[
        current_nb7["raw_unique_count"].gt(
            min(IV_CATEGORY_SETTINGS)
        ),
        "feature",
    ]
    .tolist()
)

print(
    "Categorical features with >20 raw states:",
    len(high_card_features),
)

target = meta[TARGET_COL]

iv_rows = []

for i, feature in enumerate(high_card_features, start=1):
    series = pd.read_parquet(
        TRAIN_PATH,
        columns=[feature],
        engine="pyarrow",
    )[feature].reset_index(drop=True)

    for max_categories in IV_CATEGORY_SETTINGS:
        kept_levels = learn_kept_levels(
            series,
            iv_rule_positions,
            max_categories=max_categories,
        )

        iv_value = calculate_full_iv_categorical(
            series,
            target,
            kept_levels,
            smoothing=IV_SMOOTHING,
        )

        iv_rows.append({
            "feature": feature,
            "iv_max_categories": max_categories,
            "iv": iv_value,
            "kept_level_count": len(kept_levels),
        })

    if i % 20 == 0 or i == len(high_card_features):
        print(
            f"IV sensitivity: {i} / "
            f"{len(high_card_features)}"
        )

    del series

iv_sensitivity = pd.DataFrame(iv_rows)

display(iv_sensitivity.head(20))

Categorical features with >20 raw states: 12
IV sensitivity: 12 / 12


,feature,iv_max_categories,iv,kept_level_count
0,lastapprcommoditycat_1041M,20,0.017419,20
1,lastapprcommoditycat_1041M,40,0.017631,40
2,lastapprcommoditycat_1041M,100,0.017631,40
3,lastcancelreason_561M,20,0.140557,20
4,lastcancelreason_561M,40,0.144897,40
5,lastcancelreason_561M,100,0.145434,58
6,lastrejectcommoditycat_161M,20,0.030803,20
7,lastrejectcommoditycat_161M,40,0.031424,37
8,lastrejectcommoditycat_161M,100,0.031424,37
9,previouscontdistrict_112M,20,0.018410,20


## 4. 排名与 Top-k 稳定性

除了 IV 的数值变化，进一步比较 Relevance 的排序相关性、Top-300 / Top-500 集合重叠度，以及跨越候选集边界的特征。

对于后续子集搜索，排序成员是否变化通常比单个 IV 估计值变化更直接地影响候选空间。


In [6]:
base = (
    evidence[
        [
            "feature",
            "iv",
            "mi_percentile",
            "relevance_score",
        ]
    ]
    .drop_duplicates("feature")
    .copy()
)

ranking_tables = {}
ranking_summary_rows = []

for max_categories in IV_CATEGORY_SETTINGS:
    table = base.copy()

    replacement = (
        iv_sensitivity
        .loc[
            iv_sensitivity[
                "iv_max_categories"
            ].eq(max_categories),
            ["feature", "iv"],
        ]
        .set_index("feature")["iv"]
    )

    table["iv_sensitivity"] = (
        table["feature"]
        .map(replacement)
        .fillna(table["iv"])
    )

    table["iv_percentile_sensitivity"] = (
        table["iv_sensitivity"]
        .rank(
            pct=True,
            method="average",
        )
    )

    table["relevance_sensitivity"] = (
        table[
            [
                "iv_percentile_sensitivity",
                "mi_percentile",
            ]
        ]
        .mean(axis=1)
    )

    table = (
        table
        .sort_values(
            "relevance_sensitivity",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    table["relevance_rank"] = (
        np.arange(len(table)) + 1
    )

    ranking_tables[max_categories] = table

# 20-category version作为 sensitivity baseline
baseline = (
    ranking_tables[20]
    .set_index("feature")
)

for max_categories in [40, 100]:
    alt = (
        ranking_tables[max_categories]
        .set_index("feature")
    )

    aligned = baseline[
        ["relevance_rank"]
    ].join(
        alt[["relevance_rank"]],
        lsuffix="_20",
        rsuffix=f"_{max_categories}",
    )

    spearman_rank_corr = float(
        aligned[
            "relevance_rank_20"
        ].corr(
            aligned[
                f"relevance_rank_{max_categories}"
            ],
            method="spearman",
        )
    )

    row = {
        "comparison": f"20_vs_{max_categories}",
        "spearman_rank_corr": spearman_rank_corr,
        "median_absolute_rank_shift": float(
            (
                aligned["relevance_rank_20"]
                - aligned[
                    f"relevance_rank_{max_categories}"
                ]
            )
            .abs()
            .median()
        ),
        "max_absolute_rank_shift": int(
            (
                aligned["relevance_rank_20"]
                - aligned[
                    f"relevance_rank_{max_categories}"
                ]
            )
            .abs()
            .max()
        ),
    }

    for k in TOP_K_CHECKS:
        top_base = set(
            ranking_tables[20]
            .head(k)["feature"]
        )

        top_alt = set(
            ranking_tables[max_categories]
            .head(k)["feature"]
        )

        row[f"top{k}_jaccard"] = (
            len(top_base & top_alt)
            / len(top_base | top_alt)
        )

        row[f"top{k}_entered"] = len(
            top_alt - top_base
        )

        row[f"top{k}_left"] = len(
            top_base - top_alt
        )

    ranking_summary_rows.append(row)

ranking_sensitivity_summary = pd.DataFrame(
    ranking_summary_rows
)

display(ranking_sensitivity_summary)

,comparison,spearman_rank_corr,median_absolute_rank_shift,max_absolute_rank_shift,top300_jaccard,top300_entered,top300_left,top500_jaccard,top500_entered,top500_left
0,20_vs_40,0.999797,1.0,113,0.993355,1,1,0.996008,1,1
1,20_vs_100,0.999391,1.0,202,0.993355,1,1,0.992032,2,2


In [7]:
# 看哪些 feature 排名变化最大。

rank_change = (
    ranking_tables[20][
        ["feature", "relevance_rank"]
    ]
    .rename(
        columns={
            "relevance_rank": "rank_20"
        }
    )
)

for max_categories in [40, 100]:
    temp = (
        ranking_tables[max_categories][
            ["feature", "relevance_rank"]
        ]
        .rename(
            columns={
                "relevance_rank":
                f"rank_{max_categories}"
            }
        )
    )

    rank_change = rank_change.merge(
        temp,
        on="feature",
        how="left",
    )

rank_change["shift_20_to_100"] = (
    rank_change["rank_20"]
    - rank_change["rank_100"]
)

rank_change["abs_shift_20_to_100"] = (
    rank_change["shift_20_to_100"].abs()
)

display(
    rank_change
    .sort_values(
        "abs_shift_20_to_100",
        ascending=False,
    )
    .head(30)
)

iv_sensitivity.to_csv(
    "nb8_categorical_iv_sensitivity.csv",
    index=False,
)

ranking_sensitivity_summary.to_csv(
    "nb8_ranking_sensitivity_summary.csv",
    index=False,
)

rank_change.to_csv(
    "nb8_relevance_rank_changes.csv",
    index=False,
)

,feature,rank_20,rank_40,rank_100,shift_20_to_100,abs_shift_20_to_100
307,person_main_contaddr_zipcode_807M,308,216,106,202,202
623,previouscontdistrict_112M,624,548,434,190,190
232,person_main_registaddr_zipcode_184M,233,120,71,162,162
519,person_main_contaddr_district_15M,520,407,358,162,162
466,person_main_registaddr_district_1083M,467,376,320,147,147
681,person_main_empladdr_district_926M,682,619,558,124,124
399,person_main_empladdr_zipcode_114M,400,379,355,45,45
572,lastrejectcommoditycat_161M,573,537,540,33,33
25,taxa_top_employer_code,26,18,3,23,23
1159,person2_person2_conts_role_p125_105_50_count,1160,1178,1179,-19,19


## 5. 结果判读与范围限制

- `rare_by_frequency_share`：受最小频数阈值影响、合并到 Rare 的样本比例。
- `cap_collapsed_share`：达到低频要求但因类别上限而额外合并的样本比例。
- Spearman：不同分组设置下排序的一致性。
- Top-300 / Top-500 Jaccard：关键候选集的成员重叠。
- Rank shift：个别特征优先级是否发生明显移动。

如果类别合并占比与排名变化都较小，现有规则具有较好的局部稳定性；若大量特征跨越 Top-k 边界，则需针对受影响的 IV / Relevance 证据补充验证。

**范围限制：** 本实验不重新计算 MI、Redundancy、Gain / Permutation、Interaction、完整 Temporal Evidence 或最终 Tree / LR 搜索，也不覆盖 Numeric IV 分箱数的敏感性。


## 6. 参数含义与后续审查边界

`LGBM_BIN_SAMPLE_COUNT = 50_000` 表示 LightGBM 用于构造直方图分箱边界的抽样行数，不代表“50 个 bins”或“最多 50 个类别”。

与本实验的 IV 分组直接相关的是 `IV_MAX_CATEGORIES = 20`，以及尚未在本实验重新检验的 `IV_NUMERIC_BINS = 10`。必要时可补充对 Numeric IV 分箱数 5 / 10 / 20 的独立审查。
